# Food redistribution capstone: walkthrough

Steps through each program on a small synthetic network. For the full results run `python scripts/run_all.py`.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
from foodnet.config import NetworkConfig, CATEGORIES
from foodnet.data.generator import generate_network

cfg = NetworkConfig(n_donors=15, n_recipients=10, n_volunteers=30, n_days=400)
T = generate_network(cfg)
{k: v.shape for k, v in T.items()}

## 1. The synthetic network
Each donor stocks to last week's same-day sales times a safety buffer; surplus is what's left after today's demand.

In [ ]:
s = T['sales']
s.groupby('category')['surplus_kg'].describe().round(1)

In [ ]:
s.groupby('date')['surplus_kg'].sum().plot(figsize=(9, 3), title='Total daily surplus (kg)');

## 2. Surplus forecaster
Gradient-boosted quantile regression vs seasonal naive, rolling-origin backtest.

In [ ]:
from foodnet.forecasting.surplus import build_features, rolling_backtest
feat = build_features(s)
metrics, preds = rolling_backtest(feat, n_folds=2, fold_days=14)
metrics.groupby('model')[['mae', 'wape', 'coverage_p10_p90']].mean().round(3)

## 3. Matching engine
Greedy nearest-recipient vs the optimiser on one day.

In [ ]:
from foodnet.matching.engine import greedy_nearest, optimise_allocation, allocation_summary
day = s['date'].max()
L = s[(s['date'] == day) & (s['surplus_kg'] >= 1)].merge(T['donors'][['donor_id', 'lat', 'lon']], on='donor_id')
L = L.assign(listing_id=range(len(L)), kg=L['surplus_kg'], shelf_life_h=L['category'].map(lambda c: CATEGORIES[c]['shelf_life_h']))
R = T['recipients'].merge(T['need'][T['need']['date'] == day][['recipient_id', 'need_kg']], on='recipient_id')
pd.DataFrame({'greedy': allocation_summary(greedy_nearest(L, R), L, R),
              'optimiser': allocation_summary(optimise_allocation(L, R), L, R),
              'optimiser, 15 trips': allocation_summary(optimise_allocation(L, R, max_trips=15), L, R)}).round(3)

## 4. Uncollected-listing predictor

In [ ]:
from foodnet.risk.predictor import time_split_eval
time_split_eval(T['collections']).round(3)

## 5. End-to-end simulation
See `scripts/run_all.py` for how the pieces are wired; the quick run takes about 20 seconds.

In [ ]:
import importlib.util
from pathlib import Path
spec = importlib.util.spec_from_file_location('run_all', '../scripts/run_all.py')
run_all = importlib.util.module_from_spec(spec); spec.loader.exec_module(run_all)
run_all.main(quick=True, out_dir=Path('../results_quick'), data_dir=Path('../data_quick'))